## Random Forest

In [3]:
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier

In [7]:
import pandas as pd

df_clean = pd.read_csv("../data/processed/cleaned_train_data.csv")

df_clean.head()

,case_id,Hospital_code,Hospital_type_code,City_Code_Hospital,Hospital_region_code,Available Extra Rooms in Hospital,Department,Ward_Type,Ward_Facility_Code,Bed Grade,patientid,City_Code_Patient,Type of Admission,Severity of Illness,Visitors with Patient,Age,Admission_Deposit,Stay
0,1,8,c,3,Z,3,radiotherapy,R,F,2.0,31397,7.0,Emergency,Extreme,2,51-60,4911.0,0-10
1,2,2,c,5,Z,2,radiotherapy,S,F,2.0,31397,7.0,Trauma,Extreme,2,51-60,5954.0,41-50
2,3,10,e,1,X,2,anesthesia,S,E,2.0,31397,7.0,Trauma,Extreme,2,51-60,4745.0,31-40
3,4,26,b,2,Y,2,radiotherapy,R,D,2.0,31397,7.0,Trauma,Extreme,2,51-60,7272.0,41-50
4,5,26,b,2,Y,2,radiotherapy,S,D,2.0,31397,7.0,Trauma,Extreme,2,51-60,5558.0,41-50


In [8]:
df_ml = df_clean.copy()

In [9]:
df_ml.drop(
    columns=["case_id", "patientid"],
    inplace=True
)

In [10]:
label_encoders = {}

label_columns = [
    "Age",
    "Severity of Illness"
]

for column in label_columns:
    encoder = LabelEncoder()
    df_ml[column] = encoder.fit_transform(df_ml[column])
    label_encoders[column] = encoder

In [11]:
print(df_ml["Bed Grade"].isnull().sum())

113


In [12]:
df_ml["Bed Grade"] = df_ml["Bed Grade"].fillna(
    df_ml["Bed Grade"].mode()[0]
)

In [13]:
df_ml["Bed Grade"] = df_ml["Bed Grade"].astype(int)

In [14]:
df_ml = pd.get_dummies(
    df_ml,
    columns=[
        "Hospital_type_code",
        "Hospital_region_code",
        "Department",
        "Ward_Type",
        "Ward_Facility_Code",
        "Type of Admission"
    ]
)

In [15]:
X = df_ml.drop("Stay", axis=1)

y = df_ml["Stay"]

feature_columns = X.columns.tolist()

In [16]:
print(X.shape)
print(y.shape)

(318438, 39)
(318438,)


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(254750, 39)
(63688, 39)
(254750,)
(63688,)


In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [20]:
rf_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [21]:
rf_predictions = rf_model.predict(X_test)

In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions, average="weighted")
rf_recall = recall_score(y_test, rf_predictions, average="weighted")
rf_f1 = f1_score(y_test, rf_predictions, average="weighted")

In [23]:
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")

Accuracy : 0.3784
Precision: 0.3593
Recall   : 0.3784
F1 Score : 0.3624


In [24]:
print(classification_report(y_test, rf_predictions))

                    precision    recall  f1-score   support

              0-10       0.29      0.19      0.23      4689
             11-20       0.38      0.44      0.41     15561
             21-30       0.41      0.52      0.46     17603
             31-40       0.32      0.26      0.29     10981
             41-50       0.10      0.03      0.04      2357
             51-60       0.40      0.46      0.42      7128
             61-70       0.07      0.02      0.03       554
             71-80       0.27      0.10      0.15      2031
             81-90       0.35      0.17      0.23       941
            91-100       0.25      0.07      0.11       552
More than 100 Days       0.52      0.43      0.47      1291

          accuracy                           0.38     63688
         macro avg       0.31      0.24      0.26     63688
      weighted avg       0.36      0.38      0.36     63688



In [25]:
cm = confusion_matrix(y_test, rf_predictions)
print(cm)

[[ 906 1929 1521  242   31   40   12    6    0    0    2]
 [ 927 6869 5634 1346  132  574   21   32   11    7    8]
 [ 701 5422 9149 1589  232  379   36   37   32    5   21]
 [ 330 2166 3390 2888  159 1806   22  134   20   15   51]
 [ 105  537 1048  361   67  186    5   13    6    6   23]
 [ 105  584  852 1781   53 3247   10  235  111   40  110]
 [  25   96  181   98   10   93    9   10   15    2   15]
 [  31  142  168  427    5  879    3  213   29   16  118]
 [  11   38   32  136    5  426    2   23  161    3  104]
 [   5   29   47  105    2  230    0   34    1   37   62]
 [  15   53   55  114    4  346    5   49   79   15  556]]


## XG Boost

In [ ]:
df_ml = df_clean.copy()

In [29]:
df_ml.drop(
    columns=["case_id", "patientid"],
    inplace=True
)

In [30]:
label_encoders = {}

label_columns = [
    "Age",
    "Severity of Illness"
]

for column in label_columns:
    encoder = LabelEncoder()
    df_ml[column] = encoder.fit_transform(df_ml[column])
    label_encoders[column] = encoder

In [31]:
print(df_ml["Bed Grade"].isnull().sum())

113


In [32]:
df_ml["Bed Grade"] = df_ml["Bed Grade"].fillna(
    df_ml["Bed Grade"].mode()[0]
)

In [33]:
df_ml["Bed Grade"] = df_ml["Bed Grade"].astype(int)

In [34]:
df_ml = pd.get_dummies(
    df_ml,
    columns=[
        "Hospital_type_code",
        "Hospital_region_code",
        "Department",
        "Ward_Type",
        "Ward_Facility_Code",
        "Type of Admission"
    ]
)

In [35]:
X = df_ml.drop("Stay", axis=1)

y = df_ml["Stay"]

feature_columns = X.columns.tolist()

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [37]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

In [39]:
print(y_train.head())
print(y_train.dtype)
print(y_train.unique())

231676                 21-30
166821    More than 100 Days
70566                  11-20
197982                 61-70
280389                 51-60
Name: Stay, dtype: str
str
<ArrowStringArray>
[             '21-30', 'More than 100 Days',              '11-20',
              '61-70',              '51-60',              '31-40',
               '0-10',              '71-80',              '41-50',
             '91-100',              '81-90']
Length: 11, dtype: str


In [40]:
print(df_ml["Stay"].head())
print(df_ml["Stay"].unique())

0     0-10
1    41-50
2    31-40
3    41-50
4    41-50
Name: Stay, dtype: str
<ArrowStringArray>
[              '0-10',              '41-50',              '31-40',
              '11-20',              '51-60',              '21-30',
              '71-80', 'More than 100 Days',              '81-90',
              '61-70',             '91-100']
Length: 11, dtype: str


In [41]:
from sklearn.preprocessing import LabelEncoder

stay_encoder = LabelEncoder()

df_ml["Stay"] = stay_encoder.fit_transform(df_ml["Stay"])

In [42]:
X = df_ml.drop("Stay", axis=1)
y = df_ml["Stay"]

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [45]:
xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import l

In [46]:
xgb_predictions = xgb_model.predict(X_test)

In [47]:
xgb_accuracy = accuracy_score(y_test, xgb_predictions)
xgb_precision = precision_score(y_test, xgb_predictions, average="weighted")
xgb_recall = recall_score(y_test, xgb_predictions, average="weighted")
xgb_f1 = f1_score(y_test, xgb_predictions, average="weighted")

print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")

Accuracy : 0.4225
Precision: 0.4087
Recall   : 0.4225
F1 Score : 0.3860


## CatBoost

In [3]:
from catboost import CatBoostClassifier

In [4]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=8,
    loss_function="MultiClass",
    random_seed=42,
    verbose=100
)

In [4]:
df_clean = pd.read_csv("../data/processed/cleaned_train_data.csv")


In [5]:
df_ml = df_clean.copy()

In [6]:
df_ml.drop(
    columns=["case_id", "patientid"],
    inplace=True
)

In [7]:
label_encoders = {}

label_columns = [
    "Age",
    "Severity of Illness"
]

for column in label_columns:
    encoder = LabelEncoder()
    df_ml[column] = encoder.fit_transform(df_ml[column])
    label_encoders[column] = encoder

In [8]:
print(df_ml["Bed Grade"].isnull().sum())

113


In [9]:
df_ml["Bed Grade"] = df_ml["Bed Grade"].fillna(
    df_ml["Bed Grade"].mode()[0]
)

In [10]:
df_ml["Bed Grade"] = df_ml["Bed Grade"].astype(int)

In [11]:
df_ml = pd.get_dummies(
    df_ml,
    columns=[
        "Hospital_type_code",
        "Hospital_region_code",
        "Department",
        "Ward_Type",
        "Ward_Facility_Code",
        "Type of Admission"
    ]
)

In [12]:
X = df_ml.drop("Stay", axis=1)

y = df_ml["Stay"]

feature_columns = X.columns.tolist()

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
from catboost import CatBoostClassifier

In [17]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=8,
    loss_function="MultiClass",
    random_seed=42,
    verbose=100
)

In [18]:
cat_model.fit(X_train, y_train)

0:	learn: 2.2264490	total: 701ms	remaining: 5m 49s
100:	learn: 1.4916224	total: 38.2s	remaining: 2m 30s
200:	learn: 1.4503892	total: 1m 14s	remaining: 1m 50s
300:	learn: 1.4208375	total: 1m 53s	remaining: 1m 14s
400:	learn: 1.3944105	total: 2m 31s	remaining: 37.3s
499:	learn: 1.3699247	total: 3m 14s	remaining: 0us


CatBoostClassifier(depth=8, iterations=500, learning_rate=0.1, loss_function='MultiClass', random_seed=42, verbose=100)

In [19]:
cat_predictions = cat_model.predict(X_test)

In [20]:
cat_accuracy = accuracy_score(y_test, cat_predictions)
cat_precision = precision_score(y_test, cat_predictions, average="weighted")
cat_recall = recall_score(y_test, cat_predictions, average="weighted")
cat_f1 = f1_score(y_test, cat_predictions, average="weighted")

print(f"Accuracy : {cat_accuracy:.4f}")
print(f"Precision: {cat_precision:.4f}")
print(f"Recall   : {cat_recall:.4f}")
print(f"F1 Score : {cat_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, cat_predictions))

Accuracy : 0.4257
Precision: 0.4071
Recall   : 0.4257
F1 Score : 0.3923

Classification Report:
                    precision    recall  f1-score   support

              0-10       0.39      0.17      0.24      4689
             11-20       0.43      0.51      0.47     15561
             21-30       0.43      0.65      0.52     17603
             31-40       0.41      0.24      0.30     10981
             41-50       0.19      0.00      0.01      2357
             51-60       0.42      0.49      0.45      7128
             61-70       0.00      0.00      0.00       554
             71-80       0.32      0.03      0.06      2031
             81-90       0.35      0.19      0.24       941
            91-100       0.47      0.02      0.03       552
More than 100 Days       0.52      0.42      0.46      1291

          accuracy                           0.43     63688
         macro avg       0.36      0.25      0.25     63688
      weighted avg       0.41      0.43      0.39     63688

